# Intro to Machine Learning

Machine learning is typically broken down into *supervised* and *unsupervised* learning.

We've already been doing this!

In supervised learning, we are given data with known outputs; we know the relation between input and output for the provided data.  Supervised learning problems can be separated into two classes: *regression* and *classification*.

With *regression* problems we are trying to predict results with a __continuous output__ (e.g., $y=mx+b$).  With *classification* problems, we are trying to predict results with a __discrete output__ (e.g., background vs. foreground).  To use the lingo, the input(s) (e.g., $x$) are *features*, and the outputs (e.g., $y$) are *targets*.

## The Learning Process
The general process is:
Training set -> Learning algorithm -> hypothesis

where hypothesis is a function that maps an input request to a predicted output:
x -> hypothesis -> y_prediction

For (univariate) linear regression, for example, the hypothesis is $h_\theta(x) = \theta_0 + \theta_1 x$.

### Cost Function

In order for our algorithm to "learn" it needs feedback on how good of a job it's doing, or in machine learning lingo, an *objective function*, *cost function*.

We've already defined one of those as well:
$$
J(\theta_0, \theta_1) = \frac{1}{2m}\sum_{i=1}^m(h_\theta(x^{(i)}) - y^{(i)})^2,
$$

where $m$ is the number of training points $(x^{(1)}, y^{(1)}), \ldots, (x^{(m)}, y^{(m)})$. Our training goal is to minimize the cost function $J(\theta_0, \theta_1)$.

### Gradient Decent

Let's take some time and learn a particular minimization technique: *gradient decent*.

The algorithm is pretty simple:
$$
\mathrm{repeat~until~convergence} \{\\
\theta_j = \theta_j - \alpha \frac{\partial}{\partial\theta_j}J(\theta_0, \theta_1) \mathrm{~~~~for~}j=0\mathrm{~and~}j=1
\\\}
$$

where $\alpha$ is the *learning rate*.

### Example: Univariate Linear Regression (again)

Let's use our trusty univariate linear model $h(x) = \theta_0 + \theta_1 x$ and mean squared error cost function, and perform our own regression again, this time writing our own optimization function using gradient decent.

Plugging in our linear model to the mean squared error cost function gives:
$$
J(\theta_0, \theta_1) = \frac{1}{2m}\sum_{i=1}^m(\theta_0 + \theta_1 x^{(i)} - y^{(i)})^2
$$

We'll need the partial derivatives:
$$
\frac{\partial}{\partial\theta_0}J(\theta_0, \theta_1) = \frac{1}{m}\sum_{i=1}^m(\theta_0 + \theta_1 x^{(i)} - y^{(i)}) \\
\frac{\partial}{\partial\theta_1}J(\theta_0, \theta_1) = \frac{1}{m}\sum_{i=1}^m(\theta_0 + \theta_1 x^{(i)} - y^{(i)}) x^{(i)}
$$

In [ ]:
import numpy as np
import pandas as pd
import re

from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d import axes3d

In [ ]:
gaia = pd.read_csv('../../data/gaiadr3_solar_neighborhood.csv.gz')
gaia_nearby_sel = gaia.parallax > 40.
gaia_nearby = gaia[gaia_nearby_sel]

In [ ]:
mg = gaia_nearby.mg
bp_rp = gaia_nearby.bp_rp

In [ ]:
plt.plot(bp_rp, mg, lw=0, marker=',')
plt.gca().invert_yaxis()
plt.xlabel(r"$G_\mathrm{BP} - G_\mathrm{RP}$")
plt.ylabel("$M_G$");

In [ ]:
x1, y1 = 0, 8
x2, y2 = 3, 16.5
m_ms = (y2 - y1)/(x2 - x1)
b_ms = y1 - m_ms * x1

In [ ]:
ms_sel = mg < m_ms * bp_rp + b_ms

plt.scatter(bp_rp[ms_sel], mg[ms_sel], s=1, alpha=0.5, color='r', label='Main Sequence')
plt.scatter(bp_rp[~ms_sel], mg[~ms_sel], s=1, alpha=0.5, color='k', label='Not Main Sequence')
plt.gca().invert_yaxis()
plt.title(r'$\varpi > 40$ mas')
plt.xlabel(r"$G_\mathrm{BP} - G_\mathrm{RP}$")
plt.ylabel(r"$M_G$")
plt.legend();

In [ ]:
bp_rp = bp_rp[ms_sel]
mg = mg[ms_sel]

First the obvious way.

In [ ]:
def build_hypothesis(theta_0, theta_1):
    def h(x):
        return theta_0 + theta_1 * x
    return h

def cost(theta_0, theta_1, x=bp_rp, y=mg):
    m = len(x)
    h = build_hypothesis(theta_0, theta_1)
    
    return 1/(2*m)*np.sum(np.square(h(x)-y))

In [ ]:
theta_0, theta_1 = 0, 0

In [ ]:
print(cost(theta_0, theta_1))

In [ ]:
theta_0s = [theta_0]
theta_1s = [theta_1]
costs = [cost(theta_0, theta_1)]

In [ ]:
m = len(bp_rp)
alpha = 0.01

for i in np.arange(10000):
    h = build_hypothesis(theta_0, theta_1)
    
    theta_0 -= alpha/m * np.sum(h(bp_rp) - mg)
    theta_1 -= alpha/m * np.sum((h(bp_rp) - mg) * bp_rp)
    
    theta_0s.append(theta_0)
    theta_1s.append(theta_1)
    costs.append(cost(theta_0, theta_1))

In [ ]:
fig, [ax1, ax2] = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(theta_0s, theta_1s, marker='.')
ax2.plot(costs)

ax1.set_xlabel(r'$\theta_0$')
ax1.set_ylabel(r'$\theta_1$')
ax2.set_xlabel('iterations')
ax2.set_ylabel('cost')
ax2.set_xlim(xmin=0)
# ax2.set_ylim(ymax=100)

In [ ]:
plt.plot(bp_rp, mg, lw=0, marker=',')
plt.gca().invert_yaxis()
plt.xlabel("BP-RP")
plt.ylabel("$M_G$");
plt.plot(bp_rp, build_hypothesis(theta_0, theta_1)(bp_rp));

In [ ]:
# Create grid coordinates for plotting
t0 = np.linspace(-5, 10, 100)
t1 = np.linspace(-5, 10, 100)
xx, yy = np.meshgrid(t0, t1, indexing='xy')
Z = np.zeros((t0.size, t1.size))

# Calculate Z-values (Cost) based on grid of coefficients
for (i, j), v in np.ndenumerate(Z):
    Z[i,j] = cost(xx[i, j], yy[i, j])

fig = plt.figure(figsize=(15,6))
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122, projection='3d')

# Left plot
CS = ax1.contour(xx, yy, Z, np.logspace(-2, 3, 20))
ax1.plot(theta_0s, theta_1s, color='r', marker='.')
ax1.scatter(theta_0, theta_1, color='r', marker='x')

# Right plot
ax2.plot_surface(xx, yy, Z, rstride=1, cstride=1, alpha=0.8, cmap=plt.cm.viridis)
ax2.plot(theta_0s, theta_1s, [cost(theta0, theta1) for theta0, theta1 in zip(theta_0s, theta_1s)], marker='.')
ax2.set_zlabel('Cost')
ax2.set_zlim(Z.min(),Z.max())

# settings common to both plots
for ax in fig.axes:
    ax.set_xlabel(r'$\theta_0$', fontsize=17)
    ax.set_ylabel(r'$\theta_1$', fontsize=17)

Now the less obvious way.

## Matrix-based approach

Let's say we have $m$ examples in our training set $(x^{(1)}, y^{(1)}), \ldots, (x^{(m)}, y^{(m)})$, and $n$ features.
$$
x^{(i)} =
\begin{bmatrix}
    x_0^{(i)}\\
    x_1^{(i)}\\
    x_2^{(i)}\\
    \vdots\\
    x_n^{(i)}
\end{bmatrix}
\in \mathbb{R}^{n+1}
$$

With these we can construct an $(m\times (n+1))$ *design matrix* $X$ in the following way:
$$
X =
\begin{bmatrix}
    \longleftarrow & (x^{(1)})^T & \longrightarrow  \\
    \longleftarrow & (x^{(2)})^T & \longrightarrow  \\
                   & \vdots      &                  \\
    \longleftarrow & (x^{(m)})^T & \longrightarrow  \\
\end{bmatrix}
$$

So, for example, if we only have 1 feature (in addition to $x_0=1$)
$$
x^{(i)} =
\begin{bmatrix}
    1 \\
    x_1^{(i)}
\end{bmatrix}
$$

Then our design matrix would be:
$$
X =
\begin{bmatrix}
    1 & x_1^{(1)} \\
    1 & x_1^{(2)} \\
    \vdots & \vdots \\
    1 & x_1^{(m)} \\
\end{bmatrix}
$$
and our results $y$ (really $\vec{y}$), is
$$
y =
\begin{bmatrix}
    y^{(1)} \\
    y^{(2)} \\
    \vdots \\
    y^{(m)} \\
\end{bmatrix}
$$


### Linear Regression

Using this formalism we can now write our hypothesis as

$$
h_\theta(X) = X \cdot \theta
$$

Now let's code with this formalism in mind.

In [ ]:
X = np.ones((len(bp_rp), 2))
X[:, 1] = bp_rp
y = np.array(mg).reshape((-1, 1))

print(X)
print(y)

In [ ]:
theta_0, theta_1 = 0., 0.
theta0 = np.array([[theta_0], [theta_1]])
theta = theta0.copy()
print(theta)

In [ ]:
def build_hypothesis(theta, X=X):
    def h(X):
        return X.dot(theta)
    return h

def cost(theta, X=X, y=y):
    m = X.shape[0]
    
    h = build_hypothesis(theta)
    J = 1/(2*m)*np.sum(np.square(h(X)-y))
    
    return J

In [ ]:
print(cost(theta))

In [ ]:
m = len(bp_rp)
alpha = 0.01
niter = 10000

thetas = np.empty((niter, 2))
costs = np.empty(niter)
for i in np.arange(niter):
    h = build_hypothesis(theta)
    
    theta -= alpha/m * (X.T.dot(h(X) - y))
    thetas[i] = theta.T
    costs[i] = cost(theta)

In [ ]:
fig, [ax1, ax2] = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(thetas[:, 0], thetas[:, 1])
ax2.plot(costs)

ax1.set_xlabel(r'$\theta_0$')
ax1.set_ylabel(r'$\theta_1$')
ax2.set_xlabel('iterations')
ax2.set_ylabel('cost');

In [ ]:
plt.plot(bp_rp, mg, lw=0, marker=',')
plt.gca().invert_yaxis()
plt.xlabel("BP-RP")
plt.ylabel("$M_G$");
plt.plot(bp_rp, build_hypothesis(theta)(X));

In [ ]:
# Create grid coordinates for plotting
t0 = np.linspace(-5, 10, 100)
t1 = np.linspace(-5, 10, 100)
xx, yy = np.meshgrid(t0, t1, indexing='xy')
Z = np.zeros((t0.size, t1.size))

# Calculate Z-values (Cost) based on grid of coefficients
for (i, j), v in np.ndenumerate(Z):
    Z[i,j] = cost([[xx[i, j]], [yy[i, j]]])

fig = plt.figure(figsize=(15,6))
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122, projection='3d')

# Left plot
CS = ax1.contour(xx, yy, Z, np.logspace(-2, 3, 20))
ax1.plot(thetas[:, 0], thetas[:, 1], color='r', marker='.')
ax1.scatter(theta[0], theta[1], color='r', marker='x')

# Right plot
ax2.plot_surface(xx, yy, Z, rstride=1, cstride=1, alpha=0.6, cmap=plt.cm.viridis)
ax2.set_zlabel('Cost')
ax2.set_zlim(Z.min(),Z.max())

# settings common to both plots
for ax in fig.axes:
    ax.set_xlabel(r'$\theta_0$', fontsize=17)
    ax.set_ylabel(r'$\theta_1$', fontsize=17)

We can try to be a bit smarter about this though.  If our goal is to find the minimum of the cost function $J(\theta_0, \theta_1, \ldots, \theta_n)$, and we can take derivatives of the cost function, then we can do this by solving
$$
\frac{\partial}{\partial\theta_j}J(\theta) = 0 \mathrm{~~~~(for~every~}j\mathrm{)}
$$

Doing this for the mean squared error cost function leads to
$$
\hat{\theta} = (X^TX)^{-1}X^Ty
$$
often called the *normal equation*.

Let's try it out.

In [ ]:
theta_opt = np.linalg.pinv((X.T.dot(X))).dot(X.T).dot(y)

In [ ]:
fig = plt.figure(figsize=(15,6))
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122, projection='3d')

# Left plot
CS = ax1.contour(xx, yy, Z, np.logspace(-2, 3, 20))
ax1.scatter(theta_opt[0], theta_opt[1], color='r', marker='x')
    
# Right plot
ax2.plot_surface(xx, yy, Z, rstride=1, cstride=1, alpha=0.6, cmap=plt.cm.viridis)
ax2.plot(theta_opt[0], theta_opt[1], cost(theta_opt), color='r', marker='x')
ax2.set_zlabel('Cost')
ax2.set_zlim(Z.min(),Z.max())

# settings common to both plots
for ax in fig.axes:
    ax.set_xlabel(r'$\theta_0$', fontsize=17)
    ax.set_ylabel(r'$\theta_1$', fontsize=17)

Whoa.

This is great, but doesn't scale well with $n$.  Computing $(X^TX)^{-1}$ scales like $\mathcal{O}(n^3)$.  If $n > \sim10,000$, gradient decent may be a better approach.

---
# Fitting well vs. fitting *the data you have*

Everything so far has been about making the cost function small. But a model that fits the
data you trained it on is not the goal — the goal is a model that works on data you have not
seen. Those are different things, and telling them apart requires some discipline about how
the data gets used.

## Three splits, not two

**Training set** — what the model learns from.

**Validation set** — held back while you make choices: which model, how many parameters, what
settings. You may look at it as often as you like, because you are using it to *decide*.

**Test set** — touched once, at the end, to report performance. Every time you look at the
test set and change something in response, it becomes a second validation set, and your
reported number becomes optimistic.

The rule underneath all three: **never judge a model on the data you fitted it to.** It has
already seen those answers, so its performance there measures memory, not understanding.


In [ ]:
# Split the main sequence into train / validation / test
X = bp_rp[ms_sel].values
Y = mg[ms_sel].values

rng = np.random.default_rng(1)
idx = rng.permutation(len(X))

n_train = int(0.60 * len(X))
n_val   = int(0.20 * len(X))
train, val, test = idx[:n_train], idx[n_train:n_train+n_val], idx[n_train+n_val:]

print(f'{len(X)} main-sequence stars -> train {len(train)}, val {len(val)}, test {len(test)}')


## Overfitting, made visible

The main sequence is curved, so a straight line will not capture it. We could use a
polynomial of any degree — so which?

Fit a range of degrees and watch **two** numbers: the error on the data used for fitting, and
the error on data held back. Start with a deliberately small training set of 40 stars.


In [ ]:
def rmse(coeffs, x, y):
    return np.sqrt(np.mean((np.polyval(coeffs, x) - y)**2))


small = train[:40]
degrees = [1, 2, 3, 5, 8, 12, 15]

print(f'{"degree":>7} {"train RMSE":>11} {"val RMSE":>12}')
train_err, val_err = [], []
for d in degrees:
    c = np.polyfit(X[small], Y[small], d)
    tr, va = rmse(c, X[small], Y[small]), rmse(c, X[val], Y[val])
    train_err.append(tr); val_err.append(va)
    print(f'{d:7d} {tr:11.4f} {va:12.4f}')


numpy may warn that the fit is poorly conditioned at high degree. **That warning is telling
you something real** — it is not noise to be suppressed.


In [ ]:
plt.plot(degrees, train_err, 'o-', label='training error')
plt.plot(degrees, val_err, 's-', label='validation error')
plt.yscale('log')
plt.xlabel('polynomial degree')
plt.ylabel('RMSE')
plt.legend();


**Training error falls monotonically.** It always will — more parameters can only fit the
training points more closely, and with enough of them the curve passes through every one.

**Validation error does not.** It improves, bottoms out, then climbs catastrophically. By
degree 15 the model reproduces its 40 training stars beautifully and is wrong by a factor of
a thousand on stars it has not seen. It has learned the noise.

**This is why training error is not a measure of quality.** On its own it cannot tell you
that anything has gone wrong — it looks better and better right up to the point of
uselessness.


## The same sweep, with more data


In [ ]:
print(f'{"degree":>7} {"train RMSE":>11} {"val RMSE":>12}')
for d in degrees:
    c = np.polyfit(X[train], Y[train], d)
    print(f'{d:7d} {rmse(c, X[train], Y[train]):11.4f} {rmse(c, X[val], Y[val]):12.4f}')


With the full 2,000-odd training stars, the collapse never happens. Validation error drifts
slowly downward and flattens; degree 12 is fine.

**So overfitting is not a property of the model alone.** Degree 8 was a disaster on 40 stars
and is harmless on 2,000. What matters is how many parameters you are asking the data to
determine — the ratio, not the complexity.

That has a practical consequence: *"is this model too flexible?"* is not a question you can
answer by looking at the model. You have to look at the model **and** the amount of data,
and the only reliable way to do that is to hold some back.


## So which model do you choose?

This sweep is **model comparison**: several candidates, one criterion, pick a winner. The
criterion here was held-out error, which is the most direct thing to use — it measures what
you actually care about, and it needs no assumptions.

Two things to be careful about:

- **Choosing by validation error means the validation error is now optimistic.** You picked
  the winner *because* it scored well there. Report on the test set instead.
- **Held-out error is noisy.** With a small validation set, the gap between degree 2 and
  degree 3 may be smaller than the scatter. Prefer the simpler model when the difference is
  not clearly larger than the noise.

There are alternatives that do not require holding data back — **cross-validation** reuses
every point by rotating which fold is held out, and **information criteria** (AIC, BIC, and
for Bayesian models WAIC) penalise parameter count directly. They matter most when data is
scarce enough that giving up 20% of it hurts.

**You are about to need this.** The midterm project asks you to fit this same main sequence
with a straight line and then a quadratic, and to judge which is better — that is exactly the
comparison you just made. It also asks what happens with higher-order polynomials. You now
know what to watch, and why the answer depends on how much data you kept.
